In [ ]:
import pandas as pd
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import re

# Definições globais
ARQUIVO_PLANO = '../dw/bronze/Contas_consolidado.xlsx'
EXCEL_ENTRADA = '../dw/bronze/in.xlsx'

In [ ]:
# Carrega e exibe as primeiras linhas do plano de contas processado anteriormente
df_contas = pd.read_excel(ARQUIVO_PLANO, index_col='id', dtype='object')
df_contas.tail()

Conta 13001343

In [ ]:
CONTA = '031327'
CONTA_CONTABIL = '510' #alterar variável para cód_contabil

In [ ]:
df = pd.read_excel(EXCEL_ENTRADA,  sheet_name=f'{CONTA}', index_col=False)
df.sample(15)

In [ ]:
df.rename(columns={'Histórico  ':'Histórico'}, inplace=True)
df_filtered = df[df['Histórico'].notnull()]
df_filtered = df_filtered[df_filtered['Valor'].notnull()]
df_filtered = df_filtered[['Data', 'Histórico', 'Nr. documento', 'Valor']].copy()
df_filtered[['Data']] = df_filtered[['Data']].ffill()

In [ ]:
new_columns = ['Cód. Conta Debito', 'Cód. Conta Credito', 'Cód. Histórico', 'Inicia Lote', 'Código Matriz/Filial', 'Centro de Custo Débito', 'Centro de Custo Crédito']

for column in new_columns:
    df_filtered[column] = pd.NA

df_filtered['Complemento Histórico'] = df_filtered['Histórico'] + ' - ' + df_filtered['Nr. documento']
df = df_filtered[['Data', 'Cód. Conta Debito', 'Cód. Conta Credito', 'Valor', 'Cód. Histórico', 'Complemento Histórico', 'Inicia Lote', 'Código Matriz/Filial', 'Centro de Custo Débito', 'Centro de Custo Crédito']].copy()

In [ ]:
df_filtered.head(20)

In [ ]:
df_transacoes = df_filtered.copy()

In [ ]:
#LIMPEZA: Remove números (IDs/datas) e normaliza
def limpar_texto(texto):
    if not texto or not isinstance(texto, str):
        return ""
    # Remove prefixos bancários comuns
    padroes_remover = [
        r"PAGTO DEBITO EM C/C -", 
        r"DOC - \d+", 
        r"TRANSF\. C/C \d+ PARA \d+",
        r"\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}" # Remove CNPJs se houver
    ]
    for padrao in padroes_remover:
        texto = re.sub(padrao, "", texto, flags=re.IGNORECASE)
    
    # Normalização padrão
    texto = texto.lower()
    #texto = re.sub(r'[^\w\s]', ' ', texto) # Remove pontuação
    return texto.strip()

df_contas['txt_limpo'] = df_contas['descricao'].apply(limpar_texto)
df_transacoes['txt_limpo'] = df_transacoes['Histórico'].apply(limpar_texto)

In [ ]:
df_contas_atual = df_contas.loc[df_contas['conta_bancaria'] == int(CONTA)].copy()
df_contas_atual.info()
#TODO INSERIR VALIDAÇÃO 

In [ ]:
df_contas

In [ ]:
df_transacoes.head(15)

In [ ]:
df_transacoes.loc[df_transacoes['Valor'].str.contains('-'), 'Cód. Conta Credito'] = CONTA_CONTABIL
df_transacoes.loc[~df_transacoes['Valor'].str.contains('-'), 'Cód. Conta Debito'] = CONTA_CONTABIL

df_transacoes['Valor'] = df_transacoes['Valor'].str.replace('-', '', regex=False)
df_transacoes['Valor'] = df_transacoes['Valor'].str.replace('.', '', regex=False)

df_transacoes

### Identificação por Similaridade

Em vez de um classificador tradicional (que exigiria milhares de exemplos já rotulados), usamos a Similaridade de Cosseno para comparar a transação atual com todo o seu plano de contas.

In [ ]:
# 1. PREPARAÇÃO (Fora da função)
# O Vectorizer deve "aprender" o vocabulário das contas/fornecedores
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))

# Criamos a matriz de referência APENAS com o plano de contas
# Note que aqui usamos fit_transform no df_contas
tfidf_referencia_contas = vectorizer.fit_transform(df_contas['descricao'].apply(limpar_texto))

def identificar_conta(transacao_limpa, threshold=0.65):
    if not transacao_limpa:
        return pd.Series({'Conta': 'VAZIO', 'Score': 0})

    # 2. Transforma a transação usando o vocabulário já aprendido
    v_transacao = vectorizer.transform([transacao_limpa])

    # 3. Compara com a matriz de REFERÊNCIA (contas), não com as transações
    similaridade = cosine_similarity(v_transacao, tfidf_referencia_contas)
    
    score = similaridade.max()
    idx_melhor_match = similaridade.argmax()

    if score >= threshold:
        match = df_contas.iloc[idx_melhor_match]
        return pd.Series({
            'Conta_Debito': match['conta_debito'],
            'Conta_Credito': match['conta_credito'],
            'Razão Social_Match': match['descricao'],
            'Score': round(score, 4)
        })
    
    return pd.Series({'Conta_Debito': '00', 'Conta_Credito': '00', 'conta_contabil': '00', 'Razão Social_Match': 'NÃO ENCONTRADO', 'Score': round(score, 4)})

# 4. EXECUÇÃO
# Aplica a função corrigida
resultados = df_transacoes['txt_limpo'].apply(identificar_conta)
df_final_pred = pd.concat([df_transacoes, resultados], axis=1)

In [ ]:
df_final_pred.sample(10)

In [ ]:
df_final_pred.loc[df_final_pred['Cód. Conta Debito'].isna(), 'Cód. Conta Debito'] = df_final_pred['Conta_Debito']
df_final_pred.loc[df_final_pred['Cód. Conta Credito'].isna(), 'Cód. Conta Credito'] = df_final_pred['Conta_Credito']

df_final_pred




In [ ]:
df = df_final_pred[['Data', 'Cód. Conta Debito', 'Cód. Conta Credito', 'Valor',
       'Cód. Histórico', 'Complemento Histórico', 'Inicia Lote',
       'Código Matriz/Filial', 'Centro de Custo Débito']].copy()

df

In [ ]:
linhas_vazias = df[df['Cód. Conta Debito'].isna() | df['Cód. Conta Credito'].isna()]
linhas_vazias

In [ ]:
print((df.loc[df_final_pred['Cód. Conta Credito']== '00'].shape[0] + df.loc[df_final_pred['Cód. Conta Debito']== '00'].shape[0]) / df.shape[0])

In [ ]:
df.loc[df['Cód. Conta Credito'] == '00']

In [ ]:
df.loc[df['Cód. Conta Debito'] == '00']

In [ ]:
df.to_csv(f'../dw/silver/cssm_{CONTA}_05_2026.txt', index=False, encoding='ISO-8859-1', header=False, sep=';')